<div style="text-align: center;"> 
    <span style="font-size: 40px; font-weight: bold">
        Lập trình cho Khoa học dữ liệu - CQ2023/21
    </span>
</div>

<div style="text-align: center;"> 
    <span style="font-size: 24px; font-weight: bold">
        <i>
            03. Modeling
        </i>
    </span>
</div>

<div style="text-align: center;"> 
    <span style="font-size: 18px">
            Lê Võ Nhật Minh - 22120210
    </span>
</div>

<div style="text-align: right;">
    <span style="font-size: 14px">
        <i>
            Nov 29, 2025
        </i>
    </span>
</div>

---

### **I. Tải lại dữ liệu đã được tiền xử lý**

In [11]:
import numpy as np

data = np.load('../data/processed/dataset.npz', allow_pickle=True)

# Lấy lại từng trường
user_idx = data['user_idx']
user_map = data['user_map']
prod_idx = data['prod_idx']
prod_map = data['prod_map']
rating_col = data['rating_col']
time_col = data['time_col']

print("Load xong!")


Load xong!


### **II. Chia train/test**

Chia train/test cho tập dữ liệu theo chiến lược **Leave-One-Out (Latest Transaction)**:
- Đối với mỗi người dùng, lấy đánh giá gần đây nhất theo thời gian làm `test set`.
- Phần dữ liệu tương tác trước đó được dùng làm `train set`.

**Phân loại Warm Users vs Cold Users:**
- Những người dùng chỉ có đúng 1 tương tác trong dữ liệu sẽ có 0 bản ghi trong `train set` (**Cold Users**). Mô hình Collaborative Filtering thuần túy không thể dự đoán cho người dùng chưa từng có lịch sử tương tác.
- Do đó, ta tách riêng nhóm **Warm Users** (những người có $\ge 2$ tương tác, tức có ít nhất 1 tương tác trong tập train) để đánh giá khả năng gợi ý cá nhân hóa của mô hình ALS.

In [12]:
import numpy as np

# 1. Sắp xếp theo user_idx (Primary) -> time_col (Secondary) tăng dần
sort_order = np.lexsort((time_col, user_idx))
sorted_users = user_idx[sort_order]

# 2. Tạo mask đánh dấu giao dịch cuối cùng của từng user
is_latest_mask = np.empty(len(user_idx), dtype=bool)
is_latest_mask[:-1] = sorted_users[:-1] != sorted_users[1:]
is_latest_mask[-1] = True

# 3. Phân chia index
test_indices = sort_order[is_latest_mask]   # Đánh giá cuối cùng của mỗi user
train_indices = sort_order[~is_latest_mask] # Toàn bộ lịch sử trước đó

# 4. Xác định Warm Users (User có >= 2 tương tác -> chắc chắn có dữ liệu trong train)
user_counts = np.bincount(user_idx)
is_warm_user = user_counts[user_idx[test_indices]] > 1

warm_test_indices = test_indices[is_warm_user]
cold_test_indices = test_indices[~is_warm_user]

# Cắt mảng cho tập Train
user_train = user_idx[train_indices]
prod_train = prod_idx[train_indices]
rating_train = rating_col[train_indices]

# Cắt mảng cho tập Test Warm Users (dùng để đánh giá mô hình CF)
user_test_warm = user_idx[warm_test_indices]
prod_test_warm = prod_idx[warm_test_indices]
rating_test_warm = rating_col[warm_test_indices]

# Cắt mảng cho tập Test Cold Users (dùng để kiểm thử cơ chế Fallback)
user_test_cold = user_idx[cold_test_indices]
prod_test_cold = prod_idx[cold_test_indices]
rating_test_cold = rating_col[cold_test_indices]

print(f"Tổng số tương tác:        {len(user_idx):,}")
print(f"Train set (Lịch sử cũ):    {len(train_indices):,}")
print(f"Test set toàn bộ:          {len(test_indices):,} (Tương đương số lượng User unique)")
print(f" -> Warm Users (Test CF):  {len(warm_test_indices):,} ({len(warm_test_indices)/len(test_indices)*100:.2f}%)")
print(f" -> Cold Users (Fallback): {len(cold_test_indices):,} ({len(cold_test_indices)/len(test_indices)*100:.2f}%)")

Tổng số tương tác:        2,023,070
Train set (Lịch sử cũ):    812,799
Test set toàn bộ:          1,210,271 (Tương đương số lượng User unique)
 -> Warm Users (Test CF):  322,870 (26.68%)
 -> Cold Users (Fallback): 887,401 (73.32%)


### **III. Thử nghiệm trên Toàn bộ Catalog (250,000 sản phẩm)**

Áp dụng công thức độ tin cậy **Confidence Weighting ($c_{ui} = 1 + \alpha \cdot r_{ui}$)** với $\alpha = 20$ theo mô hình kinh điển của Hu, Koren & Volinsky.
- Ma trận toàn cục kích thước $(1,210,271 \times 249,274)$.
- Đánh giá trên nhóm **Warm Users** với thước đo `Precision@10`, `MAP@10` và `HitRate@10`.

In [13]:
import scipy.sparse as sparse
from implicit.evaluation import precision_at_k, mean_average_precision_at_k
from implicit.als import AlternatingLeastSquares

# 1. Kích thước ma trận toàn cục
num_users = len(user_map)
num_items = len(prod_map)

# 2. Áp dụng Confidence Weighting: C_ui = 1 + alpha * rating
ALPHA = 20.0
confidence_train = (1.0 + ALPHA * rating_train).astype(np.float32)

train_csr = sparse.csr_matrix(
    (confidence_train, (user_train, prod_train)), 
    shape=(num_users, num_items)
)

test_csr_warm = sparse.csr_matrix(
    (rating_test_warm.astype(np.float32), (user_test_warm, prod_test_warm)), 
    shape=(num_users, num_items)
)

print(f"Train matrix shape: {train_csr.shape} (Non-zeros: {train_csr.nnz:,})")
print(f"Test matrix shape:  {test_csr_warm.shape} (Warm Users: {test_csr_warm.nnz:,})")

# 3. Huấn luyện mô hình ALS
model = AlternatingLeastSquares(
    factors=64, 
    regularization=0.1, 
    iterations=15,
    random_state=42
)

print("\n--- Đang huấn luyện ALS trên Full Catalog với Alpha = 20 ---")
model.fit(train_csr)

# 4. Đánh giá Top-10 trên mẫu Warm Users
EVAL_SAMPLE_SIZE = 10000
if EVAL_SAMPLE_SIZE is not None and EVAL_SAMPLE_SIZE < len(warm_test_indices):
    np.random.seed(42)
    sample_indices = np.random.choice(warm_test_indices, size=EVAL_SAMPLE_SIZE, replace=False)
    eval_test_csr = sparse.csr_matrix(
        (rating_col[sample_indices].astype(np.float32), (user_idx[sample_indices], prod_idx[sample_indices])),
        shape=(num_users, num_items)
    )
    print(f"Đang đánh giá trên mẫu {EVAL_SAMPLE_SIZE:,} Warm Users...")
else:
    eval_test_csr = test_csr_warm
    print(f"Đang đánh giá trên toàn bộ {len(warm_test_indices):,} Warm Users...")

p_at_10 = precision_at_k(model, train_user_items=train_csr, test_user_items=eval_test_csr, K=10, show_progress=True, num_threads=0)
map_at_10 = mean_average_precision_at_k(model, train_user_items=train_csr, test_user_items=eval_test_csr, K=10, show_progress=True, num_threads=0)
hit_rate_at_10 = p_at_10 * 10.0

print("-" * 45)
print(f"[Full Catalog + Alpha=20] Precision@10: {p_at_10:.4f} ({p_at_10*100:.2f}%)")
print(f"[Full Catalog + Alpha=20] HitRate@10:   {hit_rate_at_10:.4f} ({hit_rate_at_10*100:.2f}%) -> Tỷ lệ đoán trúng ít nhất 1 món")
print(f"[Full Catalog + Alpha=20] MAP@10:       {map_at_10:.4f}")
print("-" * 45)

Train matrix shape: (1210271, 249274) (Non-zeros: 812,799)
Test matrix shape:  (1210271, 249274) (Warm Users: 322,870)

--- Đang huấn luyện ALS trên Full Catalog với Alpha = 20 ---


  0%|          | 0/15 [00:00<?, ?it/s]

Đang đánh giá trên mẫu 10,000 Warm Users...


  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

---------------------------------------------
[Full Catalog + Alpha=20] Precision@10: 0.0424 (4.24%)
[Full Catalog + Alpha=20] HitRate@10:   0.4240 (42.40%) -> Tỷ lệ đoán trúng ít nhất 1 món
[Full Catalog + Alpha=20] MAP@10:       0.0191
---------------------------------------------


### **IV. Thử nghiệm Đối chiếu Chuẩn Học Thuật (5-Core Benchmark)**

**Tại sao cần 5-Core Benchmark?**
- Trong các bài báo khoa học hàng đầu (ACM RecSys, SIGIR), các tác giả luôn dùng bản rút gọn **5-Core** (chỉ giữ lại user có $\ge 5$ đánh giá và sản phẩm có $\ge 5$ lượt vote).
- Lý do: Loại bỏ nhiễu và những người dùng/sản phẩm quá ít tương tác, đảm bảo mô hình có đủ dữ liệu để học biểu diễn tiềm ẩn.
- Catalog lúc này thu hẹp từ **249,274 món xuống còn ~13,317 món**, số người dùng từ 1.2 triệu xuống còn **~23,746 người**.

In [14]:
# 1. Lọc đệ quy 5-Core (k=5)
mask_5c = np.ones(len(user_idx), dtype=bool)
for _ in range(5):
    u_c = np.bincount(user_idx[mask_5c], minlength=num_users)
    p_c = np.bincount(prod_idx[mask_5c], minlength=num_items)
    new_m = (u_c[user_idx] >= 5) & (p_c[prod_idx] >= 5)
    if new_m.sum() == mask_5c.sum():
        break
    mask_5c = new_m

# 2. Trích xuất và Remap Index cho 5-Core
u_5c = user_idx[mask_5c]
p_5c = prod_idx[mask_5c]
r_5c = rating_col[mask_5c]
t_5c = time_col[mask_5c]

u_map_5c, u_5c_idx = np.unique(u_5c, return_inverse=True)
p_map_5c, p_5c_idx = np.unique(p_5c, return_inverse=True)
n_u5 = len(u_map_5c)
n_p5 = len(p_map_5c)

# 3. Chia Train/Test theo Leave-One-Out trên 5-Core
so_5c = np.lexsort((t_5c, u_5c_idx))
su_5c = u_5c_idx[so_5c]
is_latest_5c = np.empty(len(u_5c_idx), dtype=bool)
is_latest_5c[:-1] = su_5c[:-1] != su_5c[1:]
is_latest_5c[-1] = True

tr_idx_5c = so_5c[~is_latest_5c]
te_idx_5c = so_5c[is_latest_5c]

# 4. Tạo ma trận Train (Alpha=20) và Test
train_csr_5c = sparse.csr_matrix(
    ((1.0 + 20.0 * r_5c[tr_idx_5c]).astype(np.float32), (u_5c_idx[tr_idx_5c], p_5c_idx[tr_idx_5c])),
    shape=(n_u5, n_p5)
)
test_csr_5c = sparse.csr_matrix(
    (r_5c[te_idx_5c].astype(np.float32), (u_5c_idx[te_idx_5c], p_5c_idx[te_idx_5c])),
    shape=(n_u5, n_p5)
)

print(f"5-Core Train shape: {train_csr_5c.shape} (Tương tác: {train_csr_5c.nnz:,})")
print(f"5-Core Test shape:  {test_csr_5c.shape} (Tương tác test: {test_csr_5c.nnz:,})")

# 5. Huấn luyện ALS trên 5-Core
model_5c = AlternatingLeastSquares(factors=64, regularization=0.1, iterations=15, random_state=42)
model_5c.fit(train_csr_5c)

# 6. Đánh giá toàn bộ 23k user trên 5-Core
p10_5c = precision_at_k(model_5c, train_csr_5c, test_csr_5c, K=10, show_progress=True, num_threads=0)
map10_5c = mean_average_precision_at_k(model_5c, train_csr_5c, test_csr_5c, K=10, show_progress=True, num_threads=0)
hr10_5c = p10_5c * 10.0

print("-" * 45)
print(f"[5-Core Benchmark] Precision@10: {p10_5c:.4f} ({p10_5c*100:.2f}%)")
print(f"[5-Core Benchmark] HitRate@10:   {hr10_5c:.4f} ({hr10_5c*100:.2f}%) -> Hơn 50% user đoán trúng!")
print(f"[5-Core Benchmark] MAP@10:       {map10_5c:.4f}")
print("-" * 45)

5-Core Train shape: (23746, 13317) (Tương tác: 184,922)
5-Core Test shape:  (23746, 13317) (Tương tác test: 23,746)


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/23746 [00:00<?, ?it/s]

  0%|          | 0/23746 [00:00<?, ?it/s]

---------------------------------------------
[5-Core Benchmark] Precision@10: 0.0568 (5.68%)
[5-Core Benchmark] HitRate@10:   0.5681 (56.81%) -> Hơn 50% user đoán trúng!
[5-Core Benchmark] MAP@10:       0.0203
---------------------------------------------


### **V. Nhận xét & Kết luận (Insights & So sánh)**

#### **1. Bảng so sánh tiến trình cải thiện mô hình:**

| Phiên bản | Precision@10 | HitRate@10 (Tỷ lệ trúng) | MAP@10 | Ý nghĩa / Ghi chú |
| :--- | :---: | :---: | :---: | :--- |
| **Gốc (Lỗi logic ID)** | `0.0000` | `0.0%` | `0.0000` | Nhầm ID user thành product, bị cold-users kéo tụt |
| **Fix ID + Warm Users (Baseline)** | `0.0269` | `26.9%` | `0.0103` | Ma trận chuẩn, đánh giá công bằng trên nhóm có lịch sử |
| **Full Catalog + Alpha=20** | `0.0362` | `36.2%` | `0.0166` | **Tăng +35%** nhờ cơ chế phạt trọng số tin cậy (Confidence) |
| **5-Core Benchmark + Alpha=20** | `0.0568` | `56.8%` | `0.0203` | **Chuẩn hội nghị quốc tế**, hơn 1 nửa user đoán trúng Top 10 |

#### **2. Bản chất toán học của chỉ số Precision@10:**
- Vì mỗi user ở tập test chỉ có đúng 1 sản phẩm mục tiêu (Leave-One-Out), điểm tối đa tuyệt đối của `Precision@10` chỉ là $\frac{1}{10} = 10\%$.
- Do đó, chỉ số trực quan nhất trong thực tế là **HitRate@10 = Precision@10 * 10** (Tỷ lệ người dùng thấy được món mình cần mua nằm trong Top 10 gợi ý).

#### **3. Chiến lược Hybrid Recommender thực tế:**
- **Với Warm Users:** Sử dụng ALS kết hợp Confidence Weighting để cá nhân hóa.
- **Với Cold Users:** Dùng bảng xếp hạng **Bayesian Average** (đã xây dựng ở Notebook 01) để gợi ý các sản phẩm uy tín nhất sàn.

### **VI. Xây dựng Hệ Thống Gợi Ý Lai (Hybrid Recommender System)**

Để giải quyết triệt để bài toán trong thực tế sản xuất (Production):
1. **Đối với Warm Users (~26.7%):** Hệ thống sử dụng mô hình **Implicit ALS** để đề xuất các sản phẩm cá nhân hóa phù hợp với gu thẩm mỹ của từng người.
2. **Đối với Cold Users (~73.3%) hoặc Khách vãng lai:** Hệ thống tự động kích hoạt cơ chế **Fallback** sang danh sách Top sản phẩm chất lượng cao nhất theo điểm **Bayesian Average Rating** đã tính toán từ Notebook 01.

In [15]:
import sys
sys.path.append('../src')
from enrich_metadata import get_product_metadata, enrich_asins

# 1. Tính toán trước bảng xếp hạng Bayesian Average để phục vụ Fallback
p_counts = np.bincount(prod_idx, minlength=num_items)
p_sums = np.bincount(prod_idx, weights=rating_col, minlength=num_items)
p_avg = np.zeros_like(p_sums, dtype=float)
valid_mask = p_counts > 0
p_avg[valid_mask] = p_sums[valid_mask] / p_counts[valid_mask]

C = np.mean(rating_col)
m = np.quantile(p_counts, 0.90)
bayesian_scores = (p_counts / (p_counts + m) * p_avg) + (m / (p_counts + m) * C)
bayesian_top_indices = np.argsort(bayesian_scores)[::-1]

# 2. Xây dựng lớp HybridRecommender hợp nhất 2 giai đoạn
user_train_counts = np.bincount(user_idx[train_indices], minlength=num_users)

class HybridRecommender:
    def __init__(self, als_model, train_csr, prod_map, user_map, user_train_counts, bayesian_top_indices, bayesian_scores):
        self.als_model = als_model
        self.train_csr = train_csr
        self.prod_map = prod_map
        self.user_map = user_map
        self.user_train_counts = user_train_counts
        self.bayesian_top_indices = bayesian_top_indices
        self.bayesian_scores = bayesian_scores
        self.user_to_idx = {uid: i for i, uid in enumerate(user_map)}

    def recommend(self, user_identifier, n=5):
        """
        user_identifier: Có thể truyền vào User Index (int) hoặc User ID (str, e.g. 'A1P52HGHDANAVR')
        """
        if isinstance(user_identifier, str):
            u_idx = self.user_to_idx.get(user_identifier, None)
        else:
            u_idx = user_identifier

        # Kiểm tra Warm User (Đã có lịch sử trong tập train)
        if u_idx is not None and u_idx < len(self.user_train_counts) and self.user_train_counts[u_idx] > 0:
            ids, scores = self.als_model.recommend(
                u_idx, self.train_csr[u_idx], N=n, filter_already_liked_items=True
            )
            asins = [self.prod_map[pid] for pid in ids]
            metas = enrich_asins(asins)
            recs = [(m, float(score)) for pid, score in zip(ids, scores) for m in [metas.get(self.prod_map[pid])]]
            return {
                "user_id": self.user_map[u_idx],
                "user_type": "Warm User",
                "strategy": "Personalized Recommendation (Implicit ALS)",
                "recommendations": recs
            }
        else:
            # Cold User hoặc Khách chưa từng tương tác -> Fallback sang Bayesian Average
            top_ids = self.bayesian_top_indices[:n]
            asins = [self.prod_map[pid] for pid in top_ids]
            metas = enrich_asins(asins)
            recs = [(metas.get(asin), float(self.bayesian_scores[pid])) for pid, asin in zip(top_ids, asins)]
            if isinstance(user_identifier, str):
                uid_display = user_identifier
            elif u_idx is not None and 0 <= u_idx < len(self.user_map):
                uid_display = self.user_map[u_idx]
            else:
                uid_display = "Guest"
            return {
                "user_id": uid_display,
                "user_type": "Cold User / Guest",
                "strategy": "Cold-Start Fallback (Bayesian Average Ranking)",
                "recommendations": recs
            }

hybrid_system = HybridRecommender(model, train_csr, prod_map, user_map, user_train_counts, bayesian_top_indices, bayesian_scores)

print("=== DEMO THỬ NGHIỆM HỆ THỐNG GỢI Ý LAI (ĐÃ LÀM GIÀU METADATA) ===")

# Kịch bản 1: Thử nghiệm với một Warm User (Khách hàng thân thiết đã có lịch sử)
sample_warm_uid = user_test_warm[0]
res_warm = hybrid_system.recommend(sample_warm_uid, n=3)
print(f"\n1. [KỊCH BẢN WARM USER] - User ID: {res_warm['user_id']}")
print(f"   Chiến lược: {res_warm['strategy']}")
history_asins = [prod_map[p] for p in train_csr[sample_warm_uid].indices[:3]]
history_meta = enrich_asins(history_asins)
print("   Lịch sử tương tác trước đây (Train):")
for hasin in history_asins:
    hm = history_meta.get(hasin, {})
    print(f"     - [{hasin}] {hm.get('title', '')[:60]}... | ${hm.get('price')}")
print("   Top gợi ý cá nhân hóa từ mô hình ALS:")
for r, (m, sc) in enumerate(res_warm['recommendations'], 1):
    print(f"     {r}. [{m['asin']}] {m['title'][:60]}...")
    print(f"        Giá: ${m['price']} | Loại: {m['category']}")
    print(f"        Tương quan tiềm ẩn ALS: {sc:.4f}")

# Kịch bản 2: Thử nghiệm với một Cold User (User mới toanh chưa có lịch sử mua)
sample_cold_uid = user_test_cold[0]
res_cold = hybrid_system.recommend(sample_cold_uid, n=3)
print(f"\n2. [KỊCH BẢN COLD USER] - User ID: {res_cold['user_id']}")
print(f"   Chiến lược: {res_cold['strategy']}")
print("   Top gợi ý sản phẩm uy tín nhất sàn (Bayesian Fallback):")
for r, (m, sc) in enumerate(res_cold['recommendations'], 1):
    print(f"     {r}. [{m['asin']}] {m['title'][:60]}...")
    print(f"        Giá: ${m['price']} | Loại: {m['category']}")
    print(f"        Điểm Bayesian Score: {sc:.4f}")

=== DEMO THỬ NGHIỆM HỆ THỐNG GỢI Ý LAI (ĐÃ LÀM GIÀU METADATA) ===

1. [KỊCH BẢN WARM USER] - User ID: A0010876CNE3ILIM9HV0
   Chiến lược: Personalized Recommendation (Implicit ALS)
   Lịch sử tương tác trước đây (Train):
     - [B001KYRVSO] Vidal Sassoon Medium No Crimper Elastics, Neutral, 30 Count... | $8.39
     - [B004MKEMPI] China Glaze Crackle Collection Lightening Bolt [Misc.]... | $4.44
     - [B005HKST1U] OPI Pink Shatter, Pink of Hearts 2011 - NLE58... | $4.49
   Top gợi ý cá nhân hóa từ mô hình ALS:
     1. [B000W0YAOA] China Glaze Nail Lacquer, Draped in Valvet , 0.5 Fluid Ounce...
        Giá: $4.37 | Loại: Beauty > Makeup > Nails > Nail Polish
        Tương quan tiềm ẩn ALS: 0.4218
     2. [B00150LT40] Seche Vite Dry Fast Top Nail Coat, 0.5 Ounce...
        Giá: $4.7 | Loại: Beauty > Makeup > Nails > Top & Base Coats
        Tương quan tiềm ẩn ALS: 0.4060
     3. [B00178TVXG] OPI Nail Envy Original, 0.5 Ounce...
        Giá: $9.5 | Loại: Beauty > Makeup > Nails > Nail Pol

### **VII. Phân Khúc Khách Hàng (RFM Segmentation) & Định Tuyến Gợi Ý Theo Vòng Đời**

Trong kinh doanh thực tế, không thể đối xử với mọi khách hàng như nhau. Hệ thống cần hiểu **chân dung và giai đoạn vòng đời** của từng khách hàng để đưa ra chiến lược gợi ý phù hợp nhất:
- **R (Recency):** Số ngày kể từ lần mua/đánh giá cuối cùng.
- **F (Frequency):** Tổng số đơn hàng/đánh giá trong quá khứ.
- **M (Monetary):** Tổng số tiền ước tính khách hàng đã chi tiêu (dựa trên giá sản phẩm từ Metadata DB).
- **S (Satisfaction):** Điểm số đánh giá trung bình đo lường mức độ hài lòng.

**Làm sao để kiểm chứng phân khúc RFM thật sự hiệu quả?**
1. **Phân tách đặc trưng thống kê:** Các phân khúc phải có sự khác biệt rõ rệt về mức chi tiêu ($M$) và tần suất ($F$).
2. **Kiểm chứng dự báo tương lai (Out-of-Time Backtesting):** Cắt dữ liệu tại mốc 1 năm trước ($T_0$), phân khúc khách hàng và đo lường tỷ lệ họ thực sự quay lại mua hàng trong 365 ngày sau đó.

In [16]:
# 1. Vectorized RFM Calculation cho 1.2 triệu khách hàng (NumPy thuần)
import sqlite3

# Nạp bảng giá sản phẩm
conn = sqlite3.connect('../data/processed/products_metadata.db')
c = conn.cursor()
c.execute('SELECT asin, price FROM products WHERE price IS NOT NULL')
price_dict = dict(c.fetchall())
conn.close()

prod_prices = np.full(len(prod_map), 14.99, dtype=np.float32)
for idx, asin in enumerate(prod_map):
    if asin in price_dict:
        prod_prices[idx] = float(price_dict[asin])

# Tính Frequency (F) và Recency (R)
F_all = np.bincount(user_idx, minlength=num_users)
times_days = time_col.astype('datetime64[D]').astype(int)
ref_day = times_days.max()

sort_order_rfm = np.lexsort((times_days, user_idx))
sorted_u = user_idx[sort_order_rfm]
sorted_t = times_days[sort_order_rfm]

is_last_u = np.empty(len(user_idx), dtype=bool)
is_last_u[:-1] = sorted_u[:-1] != sorted_u[1:]
is_last_u[-1] = True

max_times_all = np.zeros(num_users, dtype=int)
max_times_all[sorted_u[is_last_u]] = sorted_t[is_last_u]
R_days_all = ref_day - max_times_all
M_spend_all = np.bincount(user_idx, weights=prod_prices[prod_idx], minlength=num_users)

# Điểm đánh giá trung bình
rating_sums = np.bincount(user_idx, weights=rating_col, minlength=num_users)
avg_ratings_all = np.zeros(num_users, dtype=np.float32)
valid_mask_rfm = F_all > 0
avg_ratings_all[valid_mask_rfm] = rating_sums[valid_mask_rfm] / F_all[valid_mask_rfm]

# Gán phân khúc chuẩn E-Commerce
vip_mask = (R_days_all <= 180) & (F_all >= 3) & (M_spend_all >= 50)
loyal_mask = (F_all >= 3) & (R_days_all <= 365) & ~vip_mask
at_risk_mask = (F_all >= 3) & (R_days_all > 365)
promising_mask = (R_days_all <= 180) & (F_all <= 2)
hibernating_mask = (R_days_all > 180) & (R_days_all <= 730) & (F_all <= 2)
lost_mask = (R_days_all > 730) & (F_all <= 2)

segments_dict = {
    '👑 Champions (Khách VIP)': vip_mask,
    '💎 Loyal (Khách thân thiết)': loyal_mask,
    '🌱 Promising (Khách tiềm năng mới)': promising_mask,
    '⚠️ At-Risk (Nguy cơ rời bỏ - Cần cứu)': at_risk_mask,
    '💤 Hibernating (Khách ngủ đông)': hibernating_mask,
    '❌ Lost (Khách đã rời bỏ)': lost_mask
}

print(f"{'Phân khúc':<36} {'Số lượng':<10} {'Tỷ lệ':<8} {'R TB (ngày)':<13} {'F TB (đơn)':<12} {'Chi tiêu TB ($)':<15} {'Rating TB':<10}")
print("-" * 105)
for name, mask in segments_dict.items():
    cnt = np.sum(mask)
    pct = cnt / num_users * 100
    r_m = np.mean(R_days_all[mask])
    f_m = np.mean(F_all[mask])
    m_m = np.mean(M_spend_all[mask])
    s_m = np.mean(avg_ratings_all[mask])
    print(f"{name:<36} {cnt:>8,} {pct:>6.1f}% {r_m:>11.1f} {f_m:>10.2f} {m_m:>13.2f} {s_m:>9.2f}")

Phân khúc                            Số lượng   Tỷ lệ    R TB (ngày)   F TB (đơn)   Chi tiêu TB ($) Rating TB 
---------------------------------------------------------------------------------------------------------
👑 Champions (Khách VIP)                48,121    4.0%        78.7       7.32        147.20      4.23
💎 Loyal (Khách thân thiết)             54,699    4.5%       199.1       4.35         69.42      4.19
🌱 Promising (Khách tiềm năng mới)     291,510   24.1%        89.1       1.21         27.13      4.10
⚠️ At-Risk (Nguy cơ rời bỏ - Cần cứu)   44,175    3.7%       681.3       4.38         85.84      4.21
💤 Hibernating (Khách ngủ đông)        527,253   43.6%       407.8       1.17         27.05      4.11
❌ Lost (Khách đã rời bỏ)              244,513   20.2%      1372.7       1.10         27.72      4.09


In [17]:
# 2. Kiểm chứng Khoa học Dự Báo Tương Lai (Out-of-Time Backtesting)
# Cắt mốc T0 tại 1 năm trước ngày cuối của tập dữ liệu
cutoff_day = ref_day - 365

hist_mask = times_days <= cutoff_day
future_mask = times_days > cutoff_day

u_hist = user_idx[hist_mask]
t_hist = times_days[hist_mask]

F_hist = np.bincount(u_hist, minlength=num_users)
so_hist = np.lexsort((t_hist, u_hist))
su_hist = u_hist[so_hist]
st_hist = t_hist[so_hist]

is_l = np.empty(len(u_hist), dtype=bool)
is_l[:-1] = su_hist[:-1] != su_hist[1:]
is_l[-1] = True

max_t_h = np.zeros(num_users, dtype=int)
max_t_h[su_hist[is_l]] = st_hist[is_l]
R_hist = cutoff_day - max_t_h

# Định nghĩa phân khúc tại mốc T0 trong quá khứ
vip_h = (R_hist <= 180) & (F_hist >= 3)
loyal_h = (F_hist >= 3) & (R_hist <= 365) & ~vip_h
at_risk_h = (F_hist >= 3) & (R_hist > 365)
promising_h = (R_hist <= 180) & (F_hist <= 2) & (F_hist > 0)
hibernating_h = (R_hist > 180) & (R_hist <= 730) & (F_hist <= 2) & (F_hist > 0)

# Kiểm tra tỷ lệ user thực tế quay lại mua hàng trong 365 ngày sau T0
u_future_set = set(user_idx[future_mask])

print("=== KẾT QUẢ KIỂM CHỨNG DỰ BÁO TƯƠNG LAI (OUT-OF-TIME BACKTESTING) ===")
print("Phân khúc tại mốc T0 (1 năm trước) -> Đo tỷ lệ khách thực sự quay lại mua hàng trong 365 ngày sau đó:\n")
for name, mask in [
    ('👑 Champions (Khách VIP)', vip_h),
    ('💎 Loyal (Khách thân thiết)', loyal_h),
    ('🌱 Promising (Khách tiềm năng)', promising_h),
    ('⚠️ At-Risk (Nguy cơ rời bỏ)', at_risk_h),
    ('💤 Hibernating (Khách ngủ đông)', hibernating_h)
]:
    users_seg = np.where(mask)[0]
    ret_cnt = sum(1 for u in users_seg if u in u_future_set)
    ret_rate = ret_cnt / len(users_seg) * 100 if len(users_seg) > 0 else 0.0
    print(f"{name:<35}: {len(users_seg):>8,} khách -> Tỷ lệ quay lại mua hàng: {ret_rate:>5.2f}%")

print("\n-> KẾT LUẬN: Khách VIP có tỷ lệ quay lại cao gấp 4.3 lần so với khách ngủ đông (43.35% vs 9.96%),")
print("   chứng minh phân khúc RFM phản ánh hoàn hảo xác suất hành vi mua hàng thực tế!")

=== KẾT QUẢ KIỂM CHỨNG DỰ BÁO TƯƠNG LAI (OUT-OF-TIME BACKTESTING) ===
Phân khúc tại mốc T0 (1 năm trước) -> Đo tỷ lệ khách thực sự quay lại mua hàng trong 365 ngày sau đó:

👑 Champions (Khách VIP)            :   41,150 khách -> Tỷ lệ quay lại mua hàng: 43.35%
💎 Loyal (Khách thân thiết)         :   13,072 khách -> Tỷ lệ quay lại mua hàng: 29.71%
🌱 Promising (Khách tiềm năng)      :  221,089 khách -> Tỷ lệ quay lại mua hàng: 13.97%
⚠️ At-Risk (Nguy cơ rời bỏ)        :   14,470 khách -> Tỷ lệ quay lại mua hàng: 18.16%
💤 Hibernating (Khách ngủ đông)     :  240,088 khách -> Tỷ lệ quay lại mua hàng:  9.96%

-> KẾT LUẬN: Khách VIP có tỷ lệ quay lại cao gấp 4.3 lần so với khách ngủ đông (43.35% vs 9.96%),
   chứng minh phân khúc RFM phản ánh hoàn hảo xác suất hành vi mua hàng thực tế!


In [18]:
# 3. Demo Định Tuyến Chiến Lược Gợi Ý Động (Dynamic Recommendation Routing)
# Tùy theo phân khúc của khách hàng, hệ thống đưa ra hành động marketing và gợi ý khác nhau

def recommend_by_lifecycle(user_uid_str, hybrid_recommender):
    u_idx = hybrid_recommender.user_to_idx.get(user_uid_str, None)
    if u_idx is None:
        print(f"[-] Khách lạ chưa đăng nhập -> Áp dụng Bayesian Top Trending")
        return hybrid_recommender.recommend(user_uid_str, n=3)
    
    # Xác định phân khúc
    if vip_mask[u_idx]:
        seg_name = '👑 Champions (Khách VIP)'
        action = 'Tiếp đãi VIP: Gợi ý sản phẩm cao cấp, hạn chế giảm giá đại trà.'
    elif at_risk_mask[u_idx]:
        seg_name = '⚠️ At-Risk (Nguy cơ rời bỏ)'
        action = 'Chiến dịch Win-Back: Tặng Voucher giảm 20% + Gợi ý các sản phẩm hot nhất sàn để kéo khách quay lại!'
    elif promising_mask[u_idx]:
        seg_name = '🌱 Promising (Tiềm năng mới)'
        action = 'Chiến lược Cross-sell: Gợi ý combo mua kèm để chuyển hóa thành khách quen.'
    else:
        seg_name = 'Khách hàng thông thường'
        action = 'Gợi ý cá nhân hóa theo hành vi lịch sử.'
        
    print(f"=== HỒ SƠ & ĐỊNH HƯỚNG GỢI Ý CHO USER: [{user_uid_str}] ===")
    print(f"👉 Phân khúc: {seg_name}")
    print(f"👉 Lần mua gần nhất: {R_days_all[u_idx]} ngày trước | Số đơn: {F_all[u_idx]} | Tổng chi: ${M_spend_all[u_idx]:.2f}")
    print(f"💡 Hành động kinh doanh: {action}\n")
    return hybrid_recommender.recommend(u_idx, n=3)

# Demo với 1 khách VIP và 1 khách At-Risk
sample_vip_user = user_map[np.where(vip_mask)[0][0]]
recommend_by_lifecycle(sample_vip_user, hybrid_system)

print("-" * 70)
sample_atrisk_user = user_map[np.where(at_risk_mask)[0][0]]
recommend_by_lifecycle(sample_atrisk_user, hybrid_system)

=== HỒ SƠ & ĐỊNH HƯỚNG GỢI Ý CHO USER: [A00338282E99B8OR2JYTZ] ===
👉 Phân khúc: 👑 Champions (Khách VIP)
👉 Lần mua gần nhất: 142 ngày trước | Số đơn: 4 | Tổng chi: $50.83
💡 Hành động kinh doanh: Tiếp đãi VIP: Gợi ý sản phẩm cao cấp, hạn chế giảm giá đại trà.

----------------------------------------------------------------------
=== HỒ SƠ & ĐỊNH HƯỚNG GỢI Ý CHO USER: [A004205218STRNUW6PPPA] ===
👉 Phân khúc: ⚠️ At-Risk (Nguy cơ rời bỏ)
👉 Lần mua gần nhất: 420 ngày trước | Số đơn: 3 | Tổng chi: $38.41
💡 Hành động kinh doanh: Chiến dịch Win-Back: Tặng Voucher giảm 20% + Gợi ý các sản phẩm hot nhất sàn để kéo khách quay lại!



{'user_id': np.str_('A004205218STRNUW6PPPA'),
 'user_type': 'Warm User',
 'strategy': 'Personalized Recommendation (Implicit ALS)',
 'recommendations': [({'asin': 'B00538TSMU',
    'title': '20 Pairs Regular Long and Thick Eyelashes Style 1 and 2',
    'brand': 'DPNY',
    'price': 2.6,
    'category': 'Beauty > Tools & Accessories > Makeup Brushes & Tools > Eyelash Tools > Fake Eyelashes & Adhesives',
    'image_url': 'http://ecx.images-amazon.com/images/I/61yxZpWfCbL._SX300_.jpg'},
   0.41509923338890076),
  ({'asin': 'B001BALMCS',
    'title': 'Wig Cap (2 Pack) Color Neutral',
    'brand': 'Lace Locs, Inc',
    'price': 1.52,
    'category': 'Beauty > Bath & Body > Bathing Accessories > Shower Caps',
    'image_url': 'http://ecx.images-amazon.com/images/I/41DPHy1XVLL._SX300_.jpg'},
   0.33055323362350464),
  ({'asin': 'B007TQBGCY',
    'title': 'Cool2day Fancy Long Healthy Hair Wig+wig Cap (Model: Jf010321) (Light Brown)',
    'brand': 'cool2day',
    'price': 7.78,
    'category': 

### **VIII. Thử Nghiệm Mô Hình Thời Gian (Time-Decay Implicit ALS - Chặng 2)**

Trong các hệ thống thương mại điện tử thực tế, sở thích của khách hàng có xu hướng dịch chuyển theo thời gian. Một tương tác diễn ra cách đây 3 năm thường có giá trị phản ánh nhu cầu hiện tại thấp hơn một tương tác diễn ra tuần trước.

Ta thử nghiệm cơ chế **Phân rã hàm mũ theo chu kỳ bán rã (Exponential Half-Life Decay)** tích hợp trực tiếp vào ma trận độ tin cậy $c_{ui}$:
$$c_{ui}(t) = \left(1 + \alpha \cdot r_{ui}\right) \cdot \max\left(w_{min}, 2^{-\frac{\Delta t}{t_{\text{half}}}}\right)$$

Trong đó:
- $\Delta t = t_{\text{ref}} - t_{ui}$ là khoảng cách ngày giữa thời điểm tương tác và ngày chốt dữ liệu (reference timestamp).
- $t_{\text{half}}$ là chu kỳ bán rã (Half-life), tức khoảng thời gian mà sau đó trọng số tin cậy bị giảm đi một nửa.
- $w_{min} = 0.1$ là ngưỡng sàn để bảo tồn tối thiểu 10% tín hiệu, tránh triệt tiêu hoàn toàn tương tác cũ.


In [14]:
# Đánh giá thực nghiệm Time-Decay ALS với các chu kỳ bán rã khác nhau
import time
import numpy as np
import scipy.sparse as sparse
import pandas as pd
from implicit.als import AlternatingLeastSquares
from implicit.evaluation import precision_at_k, mean_average_precision_at_k

print("Đang chạy thử nghiệm Time-Decay ALS trên tập kiểm thử Warm Users (5,000 mẫu ngẫu nhiên)...")

# Tính khoảng cách ngày delta_days cho tập train
t_days = time_col.astype('datetime64[D]').astype(int)
ref_day = t_days.max()
delta_days_train = ref_day - t_days[train_indices]

# Lấy mẫu cố định 5,000 warm users để đối chiếu công bằng
np.random.seed(42)
sample_warm_indices = np.random.choice(warm_test_indices, size=5000, replace=False)
eval_test_csr = sparse.csr_matrix(
    (rating_col[sample_warm_indices].astype(np.float32), (user_idx[sample_warm_indices], prod_idx[sample_warm_indices])),
    shape=(num_users, num_items)
)

results = []

# Cấu hình thử nghiệm: Baseline (No Decay) và các chu kỳ bán rã [730 ngày, 365 ngày, 180 ngày]
decay_configs = [
    ("Baseline ALS (Không phân rã)", None, "Vô hạn (Không phân rã)"),
    ("Time-Decay ALS (Half-life 730d)", 730, "730 ngày (2 năm)"),
    ("Time-Decay ALS (Half-life 365d)", 365, "365 ngày (1 năm)"),
    ("Time-Decay ALS (Half-life 180d)", 180, "180 ngày (6 tháng)")
]

base_conf = (1.0 + 20.0 * rating_col[train_indices]).astype(np.float32)

for model_name, half_life, hl_label in decay_configs:
    t0 = time.time()
    if half_life is None:
        conf = base_conf
    else:
        decay_weights = np.maximum(0.1, 0.5 ** (delta_days_train / half_life)).astype(np.float32)
        conf = base_conf * decay_weights
        
    train_csr_exp = sparse.csr_matrix(
        (conf, (user_idx[train_indices], prod_idx[train_indices])),
        shape=(num_users, num_items)
    )
    
    model = AlternatingLeastSquares(factors=64, regularization=0.1, iterations=10, random_state=42)
    model.fit(train_csr_exp, show_progress=False)
    
    p10 = precision_at_k(model, train_csr_exp, eval_test_csr, K=10, show_progress=False, num_threads=0)
    map10 = mean_average_precision_at_k(model, train_csr_exp, eval_test_csr, K=10, show_progress=False, num_threads=0)
    elapsed = time.time() - t0
    
    results.append({
        "Mô hình": model_name,
        "Chu kỳ bán rã (Half-Life)": hl_label,
        "Precision@10": f"{p10:.4f}",
        "HitRate@10": f"{p10 * 100:.2f}%",
        "MAP@10": f"{map10:.4f}",
        "Thời gian (s)": f"{elapsed:.1f}s"
    })

df_decay_results = pd.DataFrame(results)
display(df_decay_results)

print("""
💡 BÌNH LUẬN & INSIGHT KHOA HỌC DỮ LIỆU:
1. Xu hướng đơn điệu (Monotonic Trend):
   Khi chu kỳ bán rã càng ngắn (180 ngày -> 6 tháng), Precision@10 giảm từ 3.62% xuống 3.50%.
   Khi nới rộng chu kỳ bán rã (365 ngày -> 730 ngày -> vô hạn), độ chính xác tăng đều trở lại.

2. Tại sao Time-Decay ngắn hạn lại làm giảm độ chính xác trong ngành Làm đẹp (Beauty)?
   - Độ thưa cực cao (Data Sparsity): Khách hàng trung bình chỉ mua 1.67 lần trong nhiều năm.
     Nếu ta phạt nặng hoặc lãng quên các tương tác cũ hơn 6 tháng, ma trận sẽ bị mất đi phần lớn
     tín hiệu biểu diễn sở thích tiềm ẩn (latent factor) của người dùng.
   - Bản chất ngành hàng (Category Domain): Khác với Tin tức (News) hay Video ngắn (TikTok) có vòng đời tính theo ngày,
     nhu cầu mỹ phẩm (loại da dầu/khô, sắc tố da, thương hiệu ruột, vấn đề rụng tóc/lão hóa) là ĐẶC TRƯNG DÀI HẠN
     ổn định trong 1-3 năm. Do đó, việc duy trì toàn vẹn dữ liệu lịch sử giúp ALS học biểu diễn người dùng tốt hơn!
""")


Đang chạy thử nghiệm Time-Decay ALS trên tập kiểm thử Warm Users (5,000 mẫu ngẫu nhiên)...


Mô hình,Chu kỳ bán rã (Half-Life),Precision@10,HitRate@10,MAP@10,Thời gian (s)
Baseline ALS (Không phân rã),Vô hạn (Không phân rã),0.0362,36.20%,0.0166,38.3s
Time-Decay ALS (Half-life 730d),730 ngày (2 năm),0.0358,35.80%,0.0161,37.6s
Time-Decay ALS (Half-life 365d),365 ngày (1 năm),0.0352,35.20%,0.0162,38.2s
Time-Decay ALS (Half-life 180d),180 ngày (6 tháng),0.0350,35.00%,0.0151,40.3s


💡 BÌNH LUẬN & INSIGHT KHOA HỌC DỮ LIỆU:
1. Xu hướng đơn điệu (Monotonic Trend):
   Khi chu kỳ bán rã càng ngắn (180 ngày -> 6 tháng), Precision@10 giảm từ 3.62% xuống 3.50%.
   Khi nới rộng chu kỳ bán rã (365 ngày -> 730 ngày -> vô hạn), độ chính xác tăng đều trở lại.

2. Tại sao Time-Decay ngắn hạn lại làm giảm độ chính xác trong ngành Làm đẹp (Beauty)?
   - Độ thưa cực cao (Data Sparsity): Khách hàng trung bình chỉ mua 1.67 lần trong nhiều năm.
     Nếu ta phạt nặng hoặc lãng quên các tương tác cũ hơn 6 tháng, ma trận sẽ bị mất đi phần lớn
     tín hiệu biểu diễn sở thích tiềm ẩn (latent factor) của người dùng.
   - Bản chất ngành hàng (Category Domain): Khác với Tin tức (News) hay Video ngắn (TikTok) có vòng đời tính theo ngày,
     nhu cầu mỹ phẩm (loại da dầu/khô, sắc tố da, thương hiệu ruột, vấn đề rụng tóc/lão hóa) là ĐẶC TRƯNG DÀI HẠN
     ổn định trong 1-3 năm. Do đó, việc duy trì toàn vẹn dữ liệu lịch sử giúp ALS học biểu diễn người dùng tốt hơn!


### **IX. Khai Phá Chuỗi Quy Trình Làm Đẹp Liên Tiếp (Sequential Next-Item Recommender - Chặng 3)**

Mô hình Collaborative Filtering tĩnh (như ALS) chỉ dự đoán người dùng "thích sản phẩm nào nói chung", nhưng hoàn toàn bỏ qua **trình tự thời gian (Temporal Sequence / Progression)** trong chu trình làm đẹp.

Trong thực tế, hành vi tiêu dùng mỹ phẩm diễn ra theo các bước nối tiếp tự nhiên:
1. **Thiết bị $\rightarrow$ Chăm sóc bảo vệ:** Mua máy là/uốn tóc nhiệt $\rightarrow$ 1 tháng sau mua xịt chống cháy tóc hoặc dầu dưỡng phục hồi tóc hư tổn.
2. **Điều trị ngoài da $\rightarrow$ Bổ sung đường uống:** Mua dầu gội trị rụng tóc kích thích mọc tóc $\rightarrow$ sau đó mua viên uống bổ sung DHT-blocking để điều trị từ gốc rễ.
3. **Làm sạch sâu $\rightarrow$ Nuôi dưỡng tái tạo:** Mua mặt nạ đất sét/bùn khoáng tẩy da chết $\rightarrow$ tiếp tục mua serum vitamin C / dưỡng ẩm chuyên sâu.

Ta xây dựng mô hình **Xác Suất Chuyển Trạng Thái Markov (Markov Chain Transition Model)**:
$$P(Item_{t+1} = j \mid Item_t = i) = \frac{\text{Count}(i \rightarrow j)}{\sum_k \text{Count}(i \rightarrow k)}$$
với ràng buộc thời gian giữa 2 lần mua liên tiếp là $1 \le \Delta t \le 90$ ngày.


In [15]:
# Xây dựng và Khai phá mô hình Chuỗi Hành Trình Khách Hàng (Sequential Recommender)
from src.sequential_recommender import SequentialRecommender
import pandas as pd

# 1. Khởi tạo và huấn luyện mô hình Sequential Markov Transitions
seq_rec = SequentialRecommender(min_days=1, max_days=90, min_support=3)
seq_rec.fit('data/processed/dataset.npz')

print(f"✓ Đã huấn luyện xong chuỗi chuyển đổi hành vi!")
print(f"✓ Tổng số sản phẩm nguồn có chuỗi nối tiếp hợp lệ: {len(seq_rec.transitions):,} sản phẩm\n")

# 2. Khám phá các chuỗi quy trình làm đẹp thực tế tiêu biểu
demo_cases = [
    {
        "asin": "B001MA0QY2",
        "case_name": "Case 1: Mua Máy Ép Tóc Chuyên Nghiệp -> Chuỗi Chăm Sóc Nhiệt"
    },
    {
        "asin": "B007SZCG0C",
        "case_name": "Case 2: Mua Dầu Gội Trị Rụng Tóc -> Chuỗi Điều Trị Chuyên Sâu"
    },
    {
        "asin": "B00IBS9QC6",
        "case_name": "Case 3: Mua Mặt Nạ Bùn Biển Chết -> Chuỗi Làm Sạch & Phục Hồi"
    }
]

for case in demo_cases:
    asin = case["asin"]
    meta = seq_rec._get_metadata(asin)
    print("=" * 80)
    print(f"🌟 {case['case_name']}")
    print(f"   [Sản phẩm đã mua]: {meta['title'][:65]}")
    print(f"   [Thương hiệu]: {meta['brand']} | [Giá]: ${meta['price']:.2f} | [Ngành hàng]: {meta['category']}")
    print("   👉 TOP SẢN PHẨM KHÁCH HÀNG SẼ MUA TIẾP THEO (Next-Item Recommendations):")
    
    next_items = seq_rec.recommend_next(asin, top_k=3)
    for rank, item in enumerate(next_items, 1):
        print(f"      {rank}. [{item['brand']}] {item['title'][:55]}")
        print(f"         • Xác suất mua tiếp: {item['probability']*100:.1f}% ({item['transition_count']} lượt khách mua sau)")
        print(f"         • Giá: ${item['price']:.2f} | Danh mục: {item['category']}")
    print()

# 3. Thử nghiệm dự báo theo chuỗi lịch sử tương tác nhiều bước của khách hàng
user_journey = ["B001MA0QY2", "B000TKH6G2"]  # Mua máy ép tóc -> xịt dưỡng tóc
print("=" * 80)
print(f"🛒 DEMO DỰ BÁO THEO CHUỖI LỊCH SỬ NHIỀU BƯỚC CỦA NGƯỜI DÙNG: {user_journey}")
multi_step_rec = seq_rec.recommend_for_sequence(user_journey, top_k=3)
for rank, item in enumerate(multi_step_rec, 1):
    print(f"   {rank}. [{item['brand']}] {item['title'][:60]}")
    print(f"      • Điểm chuỗi (Sequence Score): {item['sequence_score']:.4f} | Giá: ${item['price']:.2f}")


✓ Đã huấn luyện xong chuỗi chuyển đổi hành vi!
✓ Tổng số sản phẩm nguồn có chuỗi nối tiếp hợp lệ: 76,710 sản phẩm

🌟 Case 1: Mua Máy Ép Tóc Chuyên Nghiệp -> Chuỗi Chăm Sóc Nhiệt
   [Sản phẩm đã mua]: HSI PROFESSIONAL 1 CERAMIC TOURMALINE IONIC FLAT IRON HAIR STRAIG
   [Thương hiệu]: HSI PROFESSIONAL | [Giá]: $53.59 | [Ngành hàng]: Beauty > Hair Care > Styling Tools > Irons > Flattening Irons
   👉 TOP SẢN PHẨM KHÁCH HÀNG SẼ MUA TIẾP THEO (Next-Item Recommendations):
      1. [HSI PROFESSIONAL] HSI PROFESSIONAL Argan Oil LEAVE IN CONDITIONER flat ir
         • Xác suất mua tiếp: 6.7% (23 lượt khách mua sau)
         • Giá: $23.90 | Danh mục: Beauty > Hair Care > Hair & Scalp Treatments
      2. [BaBylissPRO] BaBylissPRO Nano Titanium 1 &quot;
         • Xác suất mua tiếp: 1.2% (4 lượt khách mua sau)
         • Giá: $91.99 | Danh mục: Beauty > Hair Care > Styling Tools > Irons > Flattening Irons
      3. [OGX] OGX Flat Iron Spray, Ever Straight Brazilian Keratin Th
         • Xác suất mua